# Building Decision Trees

## Goal

Build a decision tree from scratch to understand how tree-based machine learning models learn rules, evaluate splits, and make predictions.

The project starts from the simplest possible model and evolves step by step toward numerical splits, deeper trees, and boosting.

---

## Version 2

Extend the decision stump to handle both categorical and numerical features.

The model will:

- keep categorical equality rules
- introduce numerical threshold rules
- search for the best numerical threshold
- compare categorical and numerical splits
- select the split with the highest gain
- generate a new Titanic submission

In [5]:
import os
import pandas as pd

The Titanic dataset has been split in two, with half being used for training the model, and half for testing it.
Each half is loaded as a Pandas DataFrame.

In [7]:
PATH = "/kaggle/input/competitions/titanic/" if os.path.exists("/kaggle/input/competitions/titanic/") else './'

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

Review the structure of the training data.

In [8]:
train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Honorifics (Mr. Miss, etc.) may offer valuable insight, but we cannot use them in their existing form, we need to extract them into a new column.

Standardise the names to lowercase and strip whitespace.

Notice each honorific follows a comma `','` and a space `' '`, then ends with a period `'.'`, the regex below describes this pattern.

The function is called and applied to each of our existing DataFrames: `train` and `test`.

In [9]:
# Feature engineering

def add_title_feature(df):
    df = df.copy()

    names = df["Name"].str.lower().str.strip()
    df["Title"] = names.str.extract(r" ([a-z]+)\.", expand=False)

    return df

train = add_title_feature(train)
test = add_title_feature(test)

Group the columns in the dataset, our aim is to calculate the probability that someone meets our target: "survived".

The **numeric** features and **categorical** features we require from the original dataset are explicitly stated, grouped, then gathered together under **features**.

Decision trees treat numerical and categorical variables differently:
- Categorical splits: Evaluate equality rules (e.g., Sex == "female").
- Numerical splits: Evaluate threshold inequalities (e.g., Age <= 30.0).

In [10]:
# Define target and feature types

TARGET = "Survived"

NUMERIC_FEATURES = ["Age", "Fare"]
CATEGORICAL_FEATURES = ["Sex", "Title", "Pclass"]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

Review our DataFrames for missing values and count them, since decision tree algorithms usually crash if fed NaN values during mathematical comparisons

In [11]:
# Check missing values before imputation

print("Train missing values:")
print(train[FEATURES].isna().sum())

print("\nTest missing values:")
print(test[FEATURES].isna().sum())

Train missing values:
Age       177
Fare        0
Sex         0
Title       0
Pclass      0
dtype: int64

Test missing values:
Age       86
Fare       1
Sex        0
Title      0
Pclass     0
dtype: int64


Fill missing values with the median; Median is used because it is less biased by outliers, such as very high cost fares.
>Notice: Test NaNs are filled with medians from Train, so that Test data is not peaked at

In [12]:
# Fill missing numerical values using training medians

age_median = train["Age"].median()
fare_median = train["Fare"].median()

train["Age"] = train["Age"].fillna(age_median)
test["Age"] = test["Age"].fillna(age_median)

test["Fare"] = test["Fare"].fillna(fare_median)

## Data preparation

We create the training features, the target, and the test features used by the model.

This sets the stage for the split-finding algorithm.

The custom decision tree will now take `X_train` and `y_train` to evaluate splits across all 5 features `(Age, Fare, Sex, Title, Pclass)`.


In [ ]:
# Prepare training and test features

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
X_train.head()

,Age,Fare,Sex,Title,Pclass
PassengerId,,,,,
1,22.0,7.2500,male,mr,3
2,38.0,71.2833,female,mrs,1
3,26.0,7.9250,female,miss,3
4,35.0,53.1000,female,mrs,1
5,35.0,8.0500,male,mr,3


## Building the model

We now implement the decision stump from scratch, starting from the simplest prediction and gradually adding split evaluation and search.

In [18]:
# Fit a constant predictor

def fit_constant(y):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    return sum(y) / len(y)

In [19]:
# Compute mean squared error

def mean_squared_error(y, prediction):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    total_error = 0

    for value in y:
        total_error += (value - prediction) ** 2

    return total_error / len(y)

In [20]:
# Split target values using a categorical equality rule

def split_categorical(x, y, category):
    y_left = []
    y_right = []

    for feature_value, target_value in zip(x, y):
        if feature_value == category:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right

In [21]:
# Split target values using a numerical threshold

def split_numeric(x, y, threshold):
    y_left = []
    y_right = []

    for feature_value, target_value in zip(x, y):
        if feature_value <= threshold:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right

In [22]:
# Compute the weighted MSE after a categorical split

def categorical_split_mse(x, y, category):
    y_left, y_right = split_categorical(x, y, category)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [23]:
# Compute the weighted MSE after a numerical split

def numeric_split_mse(x, y, threshold):
    y_left, y_right = split_numeric(x, y, threshold)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [24]:
# Compute the error reduction produced by a categorical split

def categorical_split_gain(x, y, category):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = categorical_split_mse(x, y, category)

    return base_mse - split_mse

In [25]:
# Compute the error reduction produced by a numerical split

def numeric_split_gain(x, y, threshold):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = numeric_split_mse(x, y, threshold)

    return base_mse - split_mse

### Numerical split search

For numerical features, the model must find the threshold that produces the largest gain.

Instead of testing every possible number, we sort the unique values and test the midpoints between consecutive values.

In [26]:
# Find the best numerical split for one feature

def find_best_numeric_split(x, y):
    values = []

    for value in x:
        if value not in values:
            values.append(value)

    values.sort()

    thresholds = []

    for i in range(len(values) - 1):
        midpoint = (values[i] + values[i + 1]) / 2
        thresholds.append(midpoint)

    best_threshold = None
    best_gain = float("-inf")

    for threshold in thresholds:
        gain = numeric_split_gain(x, y, threshold)

        if gain > best_gain:
            best_threshold = threshold
            best_gain = gain

    return best_threshold, best_gain

In [27]:
# Find the best categorical split for one feature

def find_best_categorical_split(x, y):
    categories = []

    for value in x:
        if value not in categories:
            categories.append(value)

    best_category = None
    best_gain = float("-inf")

    for category in categories:
        gain = categorical_split_gain(x, y, category)

        if gain > best_gain:
            best_category = category
            best_gain = gain

    return best_category, best_gain

In [28]:
# Find the best split across all features

def find_best_feature_split(X, y):
    best_feature = None
    best_split_type = None
    best_split_value = None
    best_gain = float("-inf")

    for feature in X:
        if feature in NUMERIC_FEATURES:
            split_value, gain = find_best_numeric_split(X[feature], y)
            split_type = "numeric"
        else:
            split_value, gain = find_best_categorical_split(X[feature], y)
            split_type = "categorical"

        if gain > best_gain:
            best_feature = feature
            best_split_type = split_type
            best_split_value = split_value
            best_gain = gain

    return best_feature, best_split_type, best_split_value, best_gain

In [29]:
# Train a one-split decision stump

def fit_stump(X, y):
    best_feature, best_split_type, best_split_value, best_gain = find_best_feature_split(X, y)

    x = X[best_feature]
    if best_split_type == "categorical":
        y_left, y_right = split_categorical(x, y, best_split_value)
    else:
        y_left, y_right = split_numeric(x, y, best_split_value)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    return {
        "feature": best_feature,
        "split_type": best_split_type,
        "split_value": best_split_value,
        "left_prediction": left_prediction,
        "right_prediction": right_prediction,
        "gain": best_gain
    }

In [30]:
# Train the Version 2 model

model = fit_stump(X_train, y_train)
model

{'feature': 'Title',
 'split_type': 'categorical',
 'split_value': 'mr',
 'left_prediction': 0.15667311411992263,
 'right_prediction': 0.6978609625668449,
 'gain': 0.07133502379453835}

## Prediction and submission

The trained stump generates leaf values, converts them into binary predictions, and creates the Kaggle submission file.

In [31]:
# Predict class probabilities with the trained stump

def predict_proba_stump(X, model):
    predictions = []

    x = X[model["feature"]]

    for value in x:
        if model["split_type"] == "categorical":
            go_left = value == model["split_value"]
        else:
            go_left = value <= model["split_value"]

        if go_left:
            predictions.append(model["left_prediction"])
        else:
            predictions.append(model["right_prediction"])

    return predictions

In [32]:
# Convert probability predictions into binary classes

def predict_stump(X, model, threshold=0.5):
    probabilities = predict_proba_stump(X, model)
    predictions = []

    for probability in probabilities:
        if probability >= threshold:
            predictions.append(1)
        else:
            predictions.append(0)

    return predictions

In [33]:
# Generate binary predictions for the Kaggle test set

test_predictions = predict_stump(X_test, model)

In [34]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": test_predictions
})

submission.to_csv("submission_v2.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## Notes

This second version extends the decision stump to handle both categorical and numerical features.

- The model still uses a single split.
- Categorical features are evaluated using equality rules.
- Numerical features are evaluated using threshold rules.
- Candidate numerical thresholds are generated using midpoints between consecutive values.
- The model compares categorical and numerical splits using the same gain criterion.
- Leaf predictions are learned from the target values that reach each branch.
- Missing numerical values are filled using medians computed from the training set.
- Final Titanic predictions are converted into binary classes (`0` or `1`).

Future versions will introduce new components and gradually evolve the architecture.